# Raw Data Analysis Workflow

This notebook drives the auto-analysis flow against a single calibrated `.h5`
file that bundles **`/df`** (calibrated dld), **`/tdc`** (raw delay-line
timestamps linked to the dld via `event_group_id`), and **`/range`** (the
identified ion windows).  Such files are produced by the data-processing
notebook with `save_tdc=True` and `save_range=True`.

The detector kind (Surface Concept vs RoentDek) is auto-detected from the
linked `/tdc` group, so no manual peak typing is required when a range table
is present.  The **From range file** tab uses the loaded range table; the
**Manual ranges** tab keeps the legacy free-form entry of mc windows for
exploratory work or files without a saved range.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import subprocess
import warnings

import ipywidgets as widgets
from IPython.display import display

warnings.filterwarnings("ignore")

from pyccapt.calibration.core import share_variables
from pyccapt.calibration.tutorials.tutorials_helpers import (
    helper_auto_raw_analysis,
    helper_data_loader,
    helper_raw_data_analysis,
)

variables = share_variables.Variables()

## 1. Pick the calibrated `.h5` file

The selected file should be the bundled output of the data-processing
notebook, i.e. an `.h5` with `/df` and (ideally) `/tdc` and `/range`.

In [ ]:
button = widgets.Button(description='Load dataset')

@button.on_click
def open_file_on_click(_):
    global dataset_path
    folder_path = variables.last_directory
    script = '..//..//data_tools//run_dataset_path_qt.py'
    result = subprocess.run(
        ['python', script, folder_path, 'dataset'],
        capture_output=True,
        text=True,
        shell=False,
    )
    selected_path = result.stdout.strip()
    if selected_path and selected_path != 'No file chosen':
        dataset_path = selected_path
        variables.last_directory = dataset_path
        print(f'Selected: {dataset_path}')

button

## 2. Load the bundled file

Reads `/df` (calibrated dld) and, when present, `/tdc` (raw, linked) and
`/range` from the same `.h5`.  If the range table is in a sibling
`<dataset>_range.h5` instead, that is also picked up automatically.

In [ ]:
helper_data_loader.load_calibrated_h5(dataset_path, variables)
display(variables.data.head())
if variables.data_tdc is not None:
    display(variables.data_tdc.head())

## 3. Run analysis

Pick a tab and click *Run analysis*. Each section renders a plot followed by
an inline Markdown summary of the relevant numbers.

- **From range file** — derives the species list from the loaded `/range`
  table (no manual peak typing needed).
- **Manual ranges** — type peak windows directly; rows left at 0 are ignored.

In [ ]:
helper_auto_raw_analysis.call_auto_raw_data_analysis(variables)

## 4. (Advanced) Legacy widget workflow

The original detector-specific raw-data widget — for files that are *not* the
bundled calibrated `.h5` (e.g. raw RoentDek text files or fresh Cameca raw
imports) — is still available below.  Use it when you have not yet produced
a calibrated `.h5` from the data-processing notebook.

In [ ]:
helper_raw_data_analysis.call_raw_data_workflow(variables)